In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 9. Local Study Notes Generator

## What We're Building
A study assistant that takes raw text (textbook chapters, lecture content) and:
1. Creates **structured study notes** with headers and key points
2. Generates **flashcard Q&A pairs** for self-testing
3. Creates a **quiz** with multiple-choice questions

**You will learn:**
- Educational content generation
- Multiple extraction formats from one source
- JSON generation for structured card data
- Quiz generation with answer explanations

**Stack:** Ollama, LangChain

In [ ]:
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
import json

llm = ChatOllama(model="qwen3:8b", temperature=0.3)
print("LLM ready!")

In [ ]:
# Sample study material
study_material = """
The Human Immune System

The immune system is a complex network of cells, tissues, and organs that work together
to defend the body against harmful organisms. It can be divided into two main types:
the innate immune system and the adaptive immune system.

The Innate Immune System:
This is the body's first line of defense. It includes physical barriers like skin and
mucous membranes, as well as cells like neutrophils and macrophages. The innate system
responds quickly but non-specifically — it attacks any foreign invader the same way.
Key components include:
- Physical barriers (skin, mucous membranes)
- Chemical barriers (stomach acid, enzymes in tears)
- Cellular defenses (neutrophils, macrophages, natural killer cells)
- Inflammatory response (increased blood flow, swelling)

The Adaptive Immune System:
This system develops targeted responses to specific pathogens. It takes longer to activate
(days to weeks) but provides immunological memory. Key features:
- T cells: Directly attack infected cells (cell-mediated immunity)
- B cells: Produce antibodies (humoral immunity)
- Memory cells: Remember previous infections for faster future response
- Specificity: Each response is tailored to a particular pathogen

Antibodies are Y-shaped proteins produced by B cells. Each antibody binds to a specific
antigen (foreign molecule). Types include IgG, IgA, IgM, IgE, and IgD.

Vaccines work by introducing a weakened or inactive form of a pathogen, stimulating the
adaptive immune system to create memory cells without causing disease.
"""

print(f"Study material loaded: {len(study_material)} chars")

## Step 1 — Generate Structured Study Notes

In [ ]:
notes_prompt = ChatPromptTemplate.from_template(
    """Create structured study notes from this material.

MATERIAL:
{material}

Format the notes as:
## [Main Topic]

### [Subtopic 1]
- Key point 1
- Key point 2
  - Sub-detail
- Key point 3

### [Subtopic 2]
...

**Key Terms:**
- Term: Definition

**Remember:** [1-2 sentence memory aid or mnemonic]

Create comprehensive, well-organized study notes:"""
)

notes_chain = notes_prompt | llm | StrOutputParser()
notes = notes_chain.invoke({"material": study_material})
print("=== STUDY NOTES ===")
print(notes)

## Step 2 — Generate Flashcard Q&A Pairs

In [ ]:
flashcard_prompt = ChatPromptTemplate.from_template(
    """Create flashcard Q&A pairs from this study material.
Return ONLY a JSON array of objects.

MATERIAL:
{material}

Generate 8-10 flashcards covering key concepts. Each flashcard should have:
- "question": Clear, specific question
- "answer": Concise answer (1-3 sentences)
- "difficulty": "easy", "medium", or "hard"

Return JSON array only, no markdown:"""
)

flashcard_chain = flashcard_prompt | llm | JsonOutputParser()

flashcards = flashcard_chain.invoke({"material": study_material})
print(f"Generated {len(flashcards)} flashcards:\n")
for i, card in enumerate(flashcards, 1):
    q = card.get("question", "?")
    a = card.get("answer", "?")
    d = card.get("difficulty", "?")
    print(f"Card {i} [{d}]")
    print(f"  Q: {q}")
    print(f"  A: {a}\n")

## Step 3 — Generate a Multiple-Choice Quiz

In [ ]:
quiz_prompt = ChatPromptTemplate.from_template(
    """Create a multiple-choice quiz from this study material.
Return ONLY a JSON array.

MATERIAL:
{material}

Generate 5 questions. Each should have:
- "question": The question text
- "options": Object with keys "A", "B", "C", "D" and their text
- "correct": The correct answer letter
- "explanation": Why the correct answer is right (1-2 sentences)

Return JSON array only, no markdown:"""
)

quiz_chain = quiz_prompt | llm | JsonOutputParser()
quiz = quiz_chain.invoke({"material": study_material})

print("=== MULTIPLE CHOICE QUIZ ===\n")
for i, q in enumerate(quiz, 1):
    print(f"Question {i}: {q.get('question', '?')}")
    options = q.get("options", {})
    for letter in ["A", "B", "C", "D"]:
        if letter in options:
            print(f"  {letter}) {options[letter]}")
    print(f"  ✓ Correct: {q.get('correct', '?')}")
    print(f"  Explanation: {q.get('explanation', '')}")
    print()

In [ ]:
# Full study notes pipeline
def generate_study_pack(material: str) -> dict:
    """Generate notes, flashcards, and quiz from study material."""
    print("Generating study notes...")
    notes = notes_chain.invoke({"material": material})

    print("Generating flashcards...")
    cards = flashcard_chain.invoke({"material": material})

    print("Generating quiz...")
    quiz = quiz_chain.invoke({"material": material})

    return {"notes": notes, "flashcards": cards, "quiz": quiz}

pack = generate_study_pack(study_material)
print(f"\nStudy pack complete!")
print(f"  Notes: {len(pack['notes'])} chars")
print(f"  Flashcards: {len(pack['flashcards'])} cards")
print(f"  Quiz: {len(pack['quiz'])} questions")

## Summary & Next Steps

**What you learned:**
- ✅ Multi-format output generation from one source
- ✅ JSON structured output for flashcards and quizzes
- ✅ Educational content structuring with prompts
- ✅ Difficulty-graded content generation

**Next steps:**
- Add spaced repetition scheduling for flashcards
- Support PDF/markdown input (combine with Projects 1 & 2)
- Export flashcards to Anki format

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

prompt = PromptTemplate.from_template(
    "You are a helpful local AI assistant. Answer the user's question:\n\nQuestion: {question}\n\nAnswer:"
)

chain = prompt | llm | StrOutputParser()

# response = chain.invoke({"question": "What can you help me with?"})
# print(response)
print("LangChain inference pipeline ready!")
